# LSTM for Shampoo Sales Prediction

In this exercise we use a **Long Short-Term Memory (LSTM)** network to forecast monthly shampoo sales from a univariate time series.

**Dataset:** `shampoo.csv` -- Monthly shampoo sales over 3 years (columns: `Month`, `Sales`).

**Learning objectives:**
1. Load and visualize a univariate time series.
2. Normalize data with `MinMaxScaler`, fitted on the training months only.
3. Create sliding-window sequences for supervised learning.
4. Build and train an LSTM model in PyTorch.
5. Evaluate predictions on the test months and compare the RMSE with a naive (previous-month) baseline.

**Evaluation design.** The last 6 months are the test period. Every fitted step (here the scaler and the network) uses the earlier months only, and the test months are scored once, at the end. With 36 values there is no room for a validation period, so the settings (window length, network size, number of epochs) are fixed in advance, not tuned on the test months. Runtime: a few seconds on a laptop CPU.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# YOUR CODE HERE
# Hint: Use pd.read_csv("shampoo.csv") to load the data
# Print the shape and first few rows

In [ ]:
# YOUR CODE HERE
# Hint: Use df.describe() to get summary statistics

---
## 3. Visualize the Time Series

In [ ]:
# YOUR CODE HERE
# Hint: Use plt.plot() on the "Sales" column
# Add title, xlabel, ylabel, and grid

---
## 4. Normalize Data with MinMaxScaler

Fit the scaler on the **training months only** and apply it unchanged to every month. With a 6-month window there are 30 windows; the first 80% (24) are for training, and their inputs and targets are the first 6 + 24 = 30 months. Fitting on all 36 months would leak the range of the test months into training. Test values may then fall outside [0, 1]; that is expected, so do not refit or clip.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Extract sales values: sales = df["Sales"].values.reshape(-1, 1).astype(np.float32)
# 2. Create a MinMaxScaler(feature_range=(0, 1))
# 3. Fit it on the training months ONLY (inputs and targets of the training windows):
#       SEQ_LENGTH = 6
#       train_sequence_count = int((len(sales) - SEQ_LENGTH) * 0.8)
#       training_raw_end = SEQ_LENGTH + train_sequence_count      # = 30
#       scaler.fit(sales[:training_raw_end])
# 4. Transform ALL months with the fitted scaler: sales_scaled = scaler.transform(sales)
#    (test values may fall outside [0, 1]; do not refit or clip them)

---
## 5. Create Sequences (Sliding Window)

In [ ]:
def create_sequences(data, seq_length):
    """Create input/target pairs using a sliding window.
    
    Args:
        data: 1D array of scaled values
        seq_length: number of past time steps to use as input
    
    Returns:
        X: array of shape (num_samples, seq_length, 1)
        y: array of shape (num_samples, 1)
    """
    # YOUR CODE HERE
    # Hint:
    # Loop from i=0 to len(data)-seq_length
    #   X.append(data[i : i + seq_length])
    #   y.append(data[i + seq_length])
    # Return as numpy arrays
    pass


SEQ_LENGTH = 6  # use past 6 months to predict the next month

X, y = create_sequences(sales_scaled, SEQ_LENGTH)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

---
## 6. Train/Test Split and PyTorch DataLoaders

Each window belongs to the period that contains its **target** month. The 24 training windows have targets in months 7-30; the 6 test windows have targets in months 31-36. A test window may read months before month 31 as inputs: those values are already observed at the forecast origin, which is what a deployed forecast would see. No training target lies in the test period, and the scaler saw exactly the training months.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Split at 80%: split = int(len(X) * 0.8)
#    (check: split + SEQ_LENGTH == training_raw_end, the months the scaler saw)
# 2. X_train, X_test = X[:split], X[split:]  (same for y)
# 3. Convert to torch tensors with dtype=torch.float32
# 4. Create TensorDataset and DataLoader (batch_size=8, shuffle=True for train)

---
## 7. Build the LSTM Model

In [ ]:
class ShampooLSTM(nn.Module):
    """Simple LSTM for univariate time series forecasting."""

    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # YOUR CODE HERE
        # Hint:
        # 1. self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
        #                        batch_first=True, dropout=dropout)
        # 2. self.fc = nn.Linear(hidden_size, 1)
        pass

    def forward(self, x):
        # YOUR CODE HERE
        # Hint:
        # 1. Pass x through self.lstm to get lstm_out
        # 2. Take the last time step: last_hidden = lstm_out[:, -1, :]
        # 3. Pass through self.fc and return
        pass


model = ShampooLSTM().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 8. Train the Model

`NUM_EPOCHS = 150` is fixed in advance; it is not chosen by looking at the test months. With a validation period you would pick the epoch there, as the climate notebook does with early stopping.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. criterion = nn.MSELoss()
# 2. optimizer = optim.Adam(model.parameters(), lr=1e-3)
# 3. Loop over NUM_EPOCHS=150 epochs:
#    - model.train()
#    - For each batch: zero_grad, forward, loss, backward, step
#    - Track running loss and append epoch loss to train_losses
#    - Print every 25 epochs

---
## 9. Plot the Training Loss Curve

In [ ]:
# YOUR CODE HERE
# Hint: plt.plot(train_losses) with appropriate labels

---
## 10. Predict on Test Data and Inverse Transform

This is the final evaluation: the six test months are used once, for the model trained above.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. model.eval() and torch.no_grad()
# 2. y_pred_scaled = model(X_test_t.to(DEVICE)).cpu().numpy()
# 3. Inverse transform both predictions and actuals using scaler.inverse_transform()

In [ ]:
# YOUR CODE HERE
# Hint: Plot y_actual and y_pred on the same figure with labels and legend

---
## 11. Compute RMSE

The naive baseline repeats the last observed month and is scored on exactly the same six target months as the LSTM. Six targets are far too few to rank models: treat the comparison as a mechanics check, not as evidence that one method is better.

In [ ]:
# YOUR CODE HERE
# Hint:
# rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
# Also compute a naive baseline RMSE (predict previous value) on the same test months:
# naive_pred = scaler.inverse_transform(X_test[:, -1, :])

---
## 12. Summary

In this notebook we:

1. Loaded the shampoo sales dataset -- a simple univariate monthly time series.
2. Normalized the data with a `MinMaxScaler` fitted on the training months only.
3. Created sliding-window sequences (past 6 months to predict the next month).
4. Split the windows by target month: 24 for training, the last 6 for testing.
5. Built a 2-layer LSTM with a fully connected output head and trained it for a fixed number of epochs.
6. Predicted the test months once, inverse-transformed, and plotted results.
7. Computed RMSE and compared it with a naive baseline on the same months.

**Key takeaways:**
- LSTMs are well-suited for sequential/temporal data because they maintain memory of past inputs.
- Proper normalization is essential for stable LSTM training -- and the scaler must be fitted on training data only.
- The sliding-window approach converts a time series into a supervised learning problem.
- Always compare your model to a simple baseline (e.g., naive persistence) on the same targets; with six test months, treat the result as a mechanics check.